In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from {PDF_PATH}")
print(f"First page content: {pages[0].page_content[:500]}")  # Print first 500 characters of the first page

C:\Users\temmy\AppData\Local\Temp\ipykernel_20816\4096161422.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 9 pages from telecom_guide.pdf
First page content: Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [3]:
print(pages[2].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter (
    chunk_size = 600,
    chunk_overlap = 100,
    separators = ["\n\n", "\n", ".", ",", " "],
)
chunks =  splitter.split_documents(pages)
len(chunks)

37

In [5]:
chunks[0].page_content[:500]

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store created with {vector_store._collection.count()} vectors.")

c:\Users\temmy\OneDrive\Desktop\mlUdemy\agentic_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1379.56it/s]


Vector store created with 37 vectors.


In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VolTE and how does it improve call quality in telecom networks?"
retrieved = retriever.invoke(test_query)

In [11]:
for i, doc in enumerate(retrieved, 1):
    print(f"Chunk {i}:")
    print(doc.page_content[:500])  # Print first 500 characters of each retrieved document
    print("\n---\n")

Chunk 1:
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS (IP
Multimedia Subsystem) core. Benefits include HD voice quality (wideband audio at 16 kHz versus the 3.4 kHz of
legacy calls), fast

---

Chunk 2:
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Options > Voice & Data and select LTE. If the option is absent
the device may not support VoLTE or the profile has not been pushed to the SIM. Agents can push the VoLTE
profile r

In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: join retrieved chunks into a single string ---
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

SYSTEM_PROMPT = """You are a helpful telecom assistant that answers questions ONLY based on the provided context. If the answer is not contained within the context, respond with 'I don't know.' Do not make up answers.

context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq ---
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0.0,
    reasoning_format="parsed"
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt 
    | llm
    | StrOutputParser()
)

print("RAG CHAIN READY. ASK A QUESTION BELOW.")

RAG CHAIN READY. ASK A QUESTION BELOW.


In [25]:
question = "What time is Arsenal vs Man City match and where can I watch it?"

print(f"Question: {question}")
print("Answer:", chain.invoke(question))

Question: What time is Arsenal vs Man City match and where can I watch it?
Answer: I don't know.
